<a href="https://colab.research.google.com/github/rezaeim748/LLMs-Lab-2/blob/main/lab2_word_representations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Word Representations
**Behind the Secrets of Large Language Models**, TU Dresden, WiSe 2026/27

| Chapter | Topic |
|---|---|
| A | Tokenization |
| B | Word embeddings |
| C | Applications |

Works in Google Colab (CPU is enough) or locally. Run the setup cell first. Cells marked `# TODO` are for you, bonus exercises (★) are optional.

In [1]:
%pip install -q gensim tokenizers transformers datasets

import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import gensim.downloader as api

# A. Tokenization

What do we actually embed: words, characters or subwords? (see slides 60-74)

| Part | Topic |
|---|---|
| A1 | Words or characters? |
| A2 | BPE by hand |
| A3 | Training a BPE tokenizer |
| A4 | The GPT-2 tokenizer |

## A1. Words or characters?

In [2]:
text = "Mr. O'Neill thinks that the boys' stories about San Francisco aren't amusing."

**Exercise A1.1.** Split `text` into characters, on whitespace (`text.split()`) and with `re.findall(r"\w+|[^\w\s]", text)`. Print the number of tokens and of distinct tokens.

*Goal:* see what each method does with *Mr.*, *O'Neill*, *aren't*, *San Francisco* (see slide 61).

In [3]:
def report(name, tokens):
    print(f"{name}: {len(tokens)} tokens, {len(set(tokens))} distinct")
    print(tokens, "\n")

# TODO
char_tokens = list(text)
space_tokens = text.split()
regex_tokens = re.findall(r"\w+|[^\w\s]", text)

print(char_tokens)
print(space_tokens)
print(regex_tokens)

['M', 'r', '.', ' ', 'O', "'", 'N', 'e', 'i', 'l', 'l', ' ', 't', 'h', 'i', 'n', 'k', 's', ' ', 't', 'h', 'a', 't', ' ', 't', 'h', 'e', ' ', 'b', 'o', 'y', 's', "'", ' ', 's', 't', 'o', 'r', 'i', 'e', 's', ' ', 'a', 'b', 'o', 'u', 't', ' ', 'S', 'a', 'n', ' ', 'F', 'r', 'a', 'n', 'c', 'i', 's', 'c', 'o', ' ', 'a', 'r', 'e', 'n', "'", 't', ' ', 'a', 'm', 'u', 's', 'i', 'n', 'g', '.']
['Mr.', "O'Neill", 'thinks', 'that', 'the', "boys'", 'stories', 'about', 'San', 'Francisco', "aren't", 'amusing.']
['Mr', '.', 'O', "'", 'Neill', 'thinks', 'that', 'the', 'boys', "'", 'stories', 'about', 'San', 'Francisco', 'aren', "'", 't', 'amusing', '.']


**Exercise A1.2.** Build a vocabulary from `train_text` (lowercased) and list the words of `test_text` that are not in it.

*Goal:* see why word-level models need `<UNK>` and why *open / opened / reopened* as separate words is wasteful (see slides 62-63).

In [4]:
train_text = '''The students opened their books. The teacher opens the window.
Students in Dresden read books about language models. The model reads the text.'''
test_text = "The students reopened the windows and were reading about tokenizers."

# TODO: vocab = set of training words; oov = test words not in vocab
vocab = train_text.split()
vocab = [word.rstrip(".") for word in vocab]
test_vocab = test_text.split()
test_vocab = [word.rstrip(".") for word in vocab]
print(vocab)
print(test_vocab)

['The', 'students', 'opened', 'their', 'books', 'The', 'teacher', 'opens', 'the', 'window', 'Students', 'in', 'Dresden', 'read', 'books', 'about', 'language', 'models', 'The', 'model', 'reads', 'the', 'text']
['The', 'students', 'opened', 'their', 'books', 'The', 'teacher', 'opens', 'the', 'window', 'Students', 'in', 'Dresden', 'read', 'books', 'about', 'language', 'models', 'The', 'model', 'reads', 'the', 'text']


## A2. Byte pair encoding by hand

The toy corpus from the lecture (see slides 66-70).

In [5]:
word_freqs = {"hug": 10, "pug": 5, "pun": 12, "bun": 4, "hugs": 5}
splits = {w: list(w) for w in word_freqs}
base_vocab = sorted({c for w in word_freqs for c in w})
print("Base vocabulary:", base_vocab)
print(splits)

Base vocabulary: ['b', 'g', 'h', 'n', 'p', 's', 'u']
{'hug': ['h', 'u', 'g'], 'pug': ['p', 'u', 'g'], 'pun': ['p', 'u', 'n'], 'bun': ['b', 'u', 'n'], 'hugs': ['h', 'u', 'g', 's']}


**Exercise A2.1.** Write `pair_counts(splits)`: count all pairs of neighbouring symbols, weighted by word frequency.

*Goal:* reproduce the table on slide 67 (top pair `('u', 'g')` with 20).

In [6]:
def pair_counts(splits):
    counts = Counter()
    # TODO
    for word, symbols in splits.items():
        for i in range(len(symbols) - 1):
            pair = symbols[i] + symbols[i+1]
            counts[pair] += word_freqs[word]
    return counts

print(pair_counts(splits).most_common(5))

[('ug', 20), ('pu', 17), ('un', 16), ('hu', 15), ('gs', 5)]


**Exercise A2.2.** Write `merge(pair, splits)`, then run 3 merges: take the most frequent pair, merge it, append it to `merges` and the new symbol to `vocab_bpe`.

*Goal:* get the merges *ug*, *un*, *hug* (see slides 68-70).

In [41]:
def merge(pair, splits):
    # a, b = pair
    new_splits = {}
    # TODO: walk through the symbols of every word; when you see a followed by b, append a + b
    for word, symbols in splits.items():
        new_symbols = []
        n = len(symbols)
        i = 0
        while i in range(n-1):
            current_pair = symbols[i] + symbols[i+1]
            if current_pair == pair:
                new_symbols.append(current_pair)
                i += 1
            else:
                new_symbols.append(symbols[i])
            i += 1
        if pair != symbols[n-2] + symbols[n-1]:  # check for adding the last character
            new_symbols.append(symbols[n-1])
        new_splits[word] = new_symbols
    return new_splits


merges = []
vocab_bpe = list(base_vocab)
current = dict(splits)
# TODO: 3 merge steps, print the pair and `current` after each step
for i in range(3):
    pair = pair_counts(current).most_common(1)[0][0]
    current = merge(pair, current)
    merges.append(pair)
    vocab_bpe.append(pair)

print(current)
print(merges)

{'hug': ['hug'], 'pug': ['p', 'ug'], 'pun': ['p', 'un'], 'bun': ['b', 'un'], 'hugs': ['hug', 's']}
['ug', 'un', 'hug']


**Exercise A2.3.** Write `bpe_tokenize(word)`: start from the characters and apply `merges` in the order they were learned. Try *hugs*, *bugs*, *pugs*, *mug*.

*Goal:* see how BPE handles unseen words, and what goes wrong with *m* (see slide 71).

In [ ]:
def bpe_tokenize(word):
    # TODO
    pass

for w in ["hugs", "bugs", "pugs", "mug"]:
    print(w, bpe_tokenize(w))

## A3. Training a BPE tokenizer on real data

8,530 movie reviews from the Rotten Tomatoes dataset (used again in chapter C).

In [ ]:
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

reviews = load_dataset("cornell-movie-review-data/rotten_tomatoes")["train"]["text"]
print(len(reviews), "reviews, e.g.:", reviews[0])

**Exercise A3.1.** Train a `ByteLevelBPETokenizer` with `vocab_size` 300, 1000 and 5000 (`min_frequency=2`). For each, print the average number of tokens per word on the first 500 reviews and the tokens of `sample`.

*Goal:* see the trade-off between vocabulary size and sequence length (see slides 64 and 71).

In [ ]:
sample = "The tokenization of unbelievable movies"
# TODO
# tok = ByteLevelBPETokenizer()
# tok.train_from_iterator(reviews, vocab_size=..., min_frequency=2, show_progress=False)
# tok.encode(sample).tokens

## A4. The tokenizer of a real LLM

GPT-2 uses byte-level BPE with a vocabulary of 50,257 tokens.

In [ ]:
from transformers import AutoTokenizer
gpt2 = AutoTokenizer.from_pretrained("gpt2")
print(gpt2.vocab_size)

**Exercise A4.1.** Run `gpt2.tokenize(...)` on *tokenization*, *unbelievable*, *Dresden*, *" Dresden"* (leading space), *ChatGPT*, *2026*, *Donaudampfschifffahrtsgesellschaft*.

*Goal:* see how a real tokenizer splits words, spaces and numbers.

In [ ]:
# TODO

**Exercise A4.2.** Count the GPT-2 tokens of each sentence and divide by the English count. Add a language you speak.

*Goal:* see the token premium of languages other than English (see slides 72-73). Then compare other tokenizers at https://tiktokenizer.vercel.app/, also on a few lines of Python code: how are spaces and indentation handled?

In [ ]:
sentences = {
    "English": "The weather in Dresden is nice today, so we are going for a walk along the river.",
    "German":  "Das Wetter in Dresden ist heute schön, deshalb gehen wir am Fluss spazieren.",
    "Spanish": "Hoy hace buen tiempo en Dresde, así que vamos a pasear junto al río.",
    "Hindi":   "आज ड्रेसडेन में मौसम अच्छा है, इसलिए हम नदी के किनारे टहलने जा रहे हैं।",
    "Chinese": "今天德累斯顿天气很好，所以我们去河边散步。",
}
# TODO

**Exercise A4.3.** For each string below, print the number of characters (`len`), the Unicode code points (`ord`), the number of UTF-8 bytes (`s.encode("utf-8")`) and the GPT-2 tokens. Then decode the tokens of 👋🏼 one by one with `gpt2.decode([i])`.

*Goal:* see why GPT-2 builds its vocabulary on bytes, and that a token does not have to be a whole character (see slide 71).

In [ ]:
strings = ["a", "ß", "€", "🫨", "👋🏼", "hi there 👋🏼"]
# TODO

**Exercise A4.4.** *"How many r are in strawberry?"* is a famous question that LLMs used to get wrong. Tokenize the question and the words *strawberry* and *" strawberry"*.

*Goal:* see what the model actually receives when it is asked to count letters.

In [ ]:
question = "How many r are in strawberry?"
# TODO

## A★ Bonus (optional): your own BPE on real data

**Exercise.** Apply your `pair_counts`, `merge` and `bpe_tokenize` from A2 to the word frequencies of the first 2000 reviews (`Counter` over `review.split()`). Learn 300 merges, print the first 25 learned subwords and the compression ratio (characters / tokens over all words, weighted by frequency), and tokenize *unbelievable*, *movies*, *boring*, *tokenization*, *dresden*.

*Goal:* see which subwords BPE learns first on real text.

Hint: `pair_counts` reads the global `word_freqs`, so just assign a new one.

In [ ]:
# TODO (bonus)

# B. Word Embeddings

*"You shall know a word by the company it keeps"* (Firth, 1957). From count vectors to word2vec and pretrained GloVe (see slides 16-58).

| Part | Topic |
|---|---|
| B1 | Vectors, cosine and tf-idf |
| B2 | Training word2vec |
| B3 | Embedding magic: analogies and bias |

## B1. Vectors, cosine and tf-idf

**Exercise B1.1.** Write `cosine(u, v)`. Then compute the dot product and the cosine of `V1` with every other vector, for both sets.

*Goal:* see why similarity is measured with cosine, not the dot product (see slides 23-27).

In [ ]:
def cosine(u, v):
    # TODO
    pass

set_a = {
    "V1": [1, 1, 1, 1, 0, 0, 0, 0],
    "V2": [1, 1, 1, 1, 0, 0, 0, 0],
    "V3": [1, 1, 0, 0, 0, 0, 1, 1],
    "V4": [1, 0, 0, 0, 0, 1, 1, 1],
    "V5": [0, 0, 0, 0, 1, 1, 1, 1],
}
set_b = {"V1": [3, 4], "V2": [3, 4], "V3": [4, 3], "V4": [0, 4], "V5": [0, 3]}

# TODO

**Exercise B1.2.** Term-document matrix of four Shakespeare plays (see slides 17-20).
1. Compute the cosine between all pairs of plays.
2. Repeat without the row *good*.
3. Compute $\text{idf} = \log_{10}(N / \text{df})$ over all $N = 37$ plays (`df` below) and the tf-idf matrix with $\text{tf} = \log_{10}(\text{count} + 1)$.

*Goal:* see how a word that occurs everywhere hides the difference between comedies and histories, and how tf-idf removes it (see slides 29-33).

In [ ]:
plays = ["As You Like It", "Twelfth Night", "Julius Caesar", "Henry V"]
words = ["battle", "good", "fool", "wit"]
counts = np.array([
    [  1,  0,  7, 13],
    [114, 80, 62, 89],
    [ 36, 58,  1,  4],
    [ 20, 15,  2,  3],
])
N = 37
df = np.array([21, 37, 36, 34])  # in how many of the 37 plays each word occurs

def play_similarities(M):
    for i in range(4):
        for j in range(i + 1, 4):
            print(f"  {plays[i]:15s} ~ {plays[j]:15s} {cosine(M[:, i], M[:, j]):.2f}")

# TODO: 1. play_similarities(counts)  2. without "good"  3. idf and tf-idf

## B2. Training word2vec

Skip-gram with negative sampling (see slides 39-51), trained on **text8**: Wikipedia, lowercased, no punctuation. We use the first 5 million tokens.

In [ ]:
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus

# api.load("text8") breaks with recent versions of smart_open, so we load the file path instead
sentences = list(Text8Corpus(api.load("text8", return_path=True)))[:500]

**Exercise B2.1.** Print the number of tokens, the vocabulary size and the 10 most frequent words of `sentences`.

*Goal:* know what the model is trained on.

In [ ]:
# TODO

**Exercise B2.2.** Train `Word2Vec` with `sg=1`, `negative=5`, `window=5`, `vector_size=100`, `min_count=5`. Print the shapes of `model.wv.vectors` and `model.syn1neg`.

*Goal:* find the target matrix **W** and the context matrix **C** (see slide 50).

In [ ]:
model = None  # TODO: Word2Vec(sentences=sentences, ..., workers=2, seed=42)

**Exercise B2.3.** Use `model.wv.most_similar` for *car*, *computer*, *germany*, *dresden*, and `model.wv.similarity` for *football / basketball* vs *football / bottle*. Then try *apricot*.

*Goal:* check that the neighbours make sense, and meet the OOV problem again.

In [ ]:
# TODO

**Exercise B2.4.** Try *king - man + woman* and *paris - france + germany* with `model.wv.most_similar(positive=[...], negative=[...])`.

*Goal:* see which analogies a model trained on 5M tokens can solve (see slide 54).

In [ ]:
# TODO

**Exercise B2.5 (optional).** Retrain on `sentences[:200]` with a different `window` (2 vs 10), `vector_size` or `sg=0` (CBOW) and compare the neighbours of a few words.

*Goal:* get a feeling for the hyperparameters.

In [ ]:
# TODO (optional)

## B3. Embedding magic with pretrained GloVe

50-dimensional vectors for 400,000 words, trained on 6 billion tokens.

In [ ]:
glove = api.load("glove-wiki-gigaword-50")
print(glove["dresden"][:10])

**Exercise B3.1.** Use `glove.most_similar` for *bank*, *apple*, *python*, *mouse*.

*Goal:* see that a static embedding mixes all senses of a word (see slide 7).

In [ ]:
# TODO

**Exercise B3.2.** Implement `analogy(a, b, c)`: the word closest to `b - a + c`, excluding `a`, `b`, `c` (use `glove.similar_by_vector`). Test the cases below, then find two analogies that fail.

*Goal:* reproduce the parallelogram method (see slides 53-54).

In [ ]:
def analogy(a, b, c, topn=1):
    # TODO
    pass

tests = [("apple", "tree", "grape"), ("man", "king", "woman"), ("france", "paris", "italy"),
         ("walk", "walked", "swim"), ("good", "better", "bad")]
# TODO: print analogy(*t) for every test, then try your own

**Exercise B3.3.** Run *father : doctor :: mother : ?*. Then compute the cosine of each profession with the gender direction `g = glove["he"] - glove["she"]` (normalised) and sort.

*Goal:* see that embeddings copy stereotypes from their training data (see slide 57).

In [ ]:
professions = ["nurse", "engineer", "doctor", "teacher", "programmer", "secretary", "pilot",
               "dancer", "scientist", "receptionist", "manager", "librarian", "surgeon", "housekeeper"]
# TODO

**Exercise B3.4.** Plot the words below in 2D with PCA, one colour per group.

*Goal:* see whether the groups form clusters. Interactive 3D version: https://projector.tensorflow.org/

In [ ]:
from sklearn.decomposition import PCA

groups = {
    "animals": ["cat", "dog", "horse", "mouse", "lion"],
    "cities": ["dresden", "berlin", "paris", "london", "tokyo"],
    "food": ["bread", "cheese", "apple", "rice", "soup"],
    "verbs": ["run", "walk", "swim", "jump", "eat"],
}
# TODO

## B★ Bonus (optional): removing the gender direction

**Exercise.** Write `debias(v)` that returns `v - (v @ g) * g`. For *nurse*, *engineer*, *receptionist*, *manager*, compare the cosine with `g`, *she* and *he* before and after. Then repeat *father : doctor :: mother : ?* with debiased vectors for all three words.

*Goal:* see how much this simple fix (Bolukbasi et al., 2016) removes, and what remains.

In [ ]:
# TODO (bonus)

# C. Applications

Using word vectors for two tasks: sentence similarity and sentiment classification.

| Part | Topic |
|---|---|
| C1 | Sentence similarity with averaged word vectors |
| C2 | Sentiment classification: dense vs sparse features |

## C1. Sentence similarity

**Exercise C1.1.** Write
- `preprocess(text)`: lowercase and return the words (`re.findall(r"[a-z]+(?:'[a-z]+)?", ...)`),
- `sentence_vector(text)`: the mean GloVe vector of all known words (zeros if none),
- `cosine(u, v)`: 0 if one of the vectors is zero.

*Goal:* a simple sentence embedding.

In [ ]:
def preprocess(text):
    # TODO
    pass

def sentence_vector(text):
    # TODO
    pass

def cosine(u, v):
    # TODO
    pass

**Exercise C1.2.** For each query, print the 3 most similar sentences with their scores. Add your own query.

*Goal:* semantic search with averaged word vectors.

In [ ]:
sentences = [
    "The tram to the main station is delayed again",
    "Our bus was late because of the snow",
    "The library opens at eight in the morning",
    "Students can borrow books until midnight",
    "The football club won the final match",
    "The team celebrated an unexpected victory",
    "Neural networks learn representations from data",
    "Language models predict the next word",
    "I baked fresh bread with my grandmother",
    "The soup needs more salt and pepper",
]
queries = ["public transport problems", "a machine learning lecture", "cooking dinner"]

# TODO

**Exercise C1.3.** Plot the cosine similarity matrix of the 10 sentences (`plt.imshow`).

*Goal:* see whether sentences on the same topic form blocks.

In [ ]:
# TODO

**Exercise C1.4.** Compute the cosine for the pairs below.

*Goal:* find out what averaging throws away.

In [ ]:
pairs = [
    ("the dog bit the man", "the man bit the dog"),
    ("the movie was good", "the movie was not good"),
    ("the movie was good", "the film was great"),
]
# TODO

## C2. Sentiment classification

Rotten Tomatoes: 8,530 training and 1,066 test reviews, one sentence each, labelled positive (1) or negative (0).

In [ ]:
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_texts, train_labels = data["train"]["text"], np.array(data["train"]["label"])
test_texts, test_labels = data["test"]["text"], np.array(data["test"]["label"])
print(len(train_texts), "train,", len(test_texts), "test")
print(train_texts[0], train_labels[0])

**Exercise C2.1.** Turn every review into its averaged GloVe vector (`X_train`, `X_test`), train `LogisticRegression(max_iter=1000)` and print the test accuracy.

*Goal:* a classifier on dense features.

In [ ]:
# TODO

**Exercise C2.2.** Do the same with `TfidfVectorizer` (fit on the training texts only). Print the shape of the training matrix and the test accuracy.

*Goal:* compare dense with sparse features (see slides 30-36).

In [ ]:
# TODO

**Exercise C2.3.** Train both classifiers again on only the 100 reviews in `idx` and compare the test accuracy.

*Goal:* see when dense vectors win (see slide 36).

In [ ]:
idx = np.random.RandomState(0).permutation(len(train_texts))[:100]
small_texts = [train_texts[i] for i in idx]
small_labels = train_labels[idx]
# TODO

**Exercise C2.4.** Classify your own reviews with both full models. Try negation and sarcasm.

*Goal:* find the limits of both approaches.

In [ ]:
my_reviews = [
    "I absolutely loved this movie, it was fantastic!",
    "Terrible. Waste of time.",
    "Not bad at all, actually quite enjoyable.",
    "Oh great, another two hours of my life I will never get back.",
]
# TODO

## C★ Bonus (optional): better dense features

**Exercise.** Improve the GloVe classifier in two ways and compare the test accuracy:
1. an idf-weighted average (weights from `dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))`, weight 1 for unknown words),
2. a plain average of `glove-wiki-gigaword-100` vectors.

*Goal:* see which helps more, and whether either beats tf-idf.

In [ ]:
# TODO (bonus)